In [0]:
%sql
CREATE TABLE IF NOT EXISTS de_learning_workspace.silver.order_items
(
    order_item_id INT,
    order_id INT,
    product_id INT,

    quantity INT,
    unit_price DECIMAL(10,2),
    discount DECIMAL(10,2),

    line_amount DECIMAL(12,2),

    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    _load_timestamp TIMESTAMP,
    _source_system STRING,

    _silver_processed_timestamp TIMESTAMP
)
USING DELTA;

In [0]:
from pyspark.sql.functions import *

bronze_df = spark.table("de_learning_workspace.bronze.order_items")

silver_df = (
    bronze_df

    .dropDuplicates(["order_item_id"])

    .filter(col("quantity") > 0)

    .filter(col("unit_price") >= 0)

    .withColumn(
        "line_amount",
        ((col("quantity") * col("unit_price")) - col("discount")).cast("decimal(12,2)")
    )

    .withColumn(
        "_silver_processed_timestamp",
        current_timestamp()
    )
)

(
    silver_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("de_learning_workspace.silver.order_items")
)